# 05 — Backpropagation From Scratch

Backpropagation is an efficient application of the **chain rule** through a computational graph.

Forward pass computes values left → right. Backward pass propagates sensitivity right → left.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=6,suppress=True)
def sigmoid(z): return 1/(1+np.exp(-z))

## 1. Tiny network: 1 input → 1 hidden neuron → 1 output

In [ ]:
x=2.0; y=1.0
w1,b1=.5,.1
w2,b2=-.7,.2
z1=w1*x+b1
a1=np.tanh(z1)
z2=w2*a1+b2
yhat=sigmoid(z2)
loss=-(y*np.log(yhat)+(1-y)*np.log(1-yhat))
print(dict(z1=z1,a1=a1,z2=z2,yhat=yhat,loss=loss))

## 2. Backward pass, derivative by derivative

In [ ]:
dL_dz2=yhat-y
dL_dw2=dL_dz2*a1
dL_db2=dL_dz2
dL_da1=dL_dz2*w2
da1_dz1=1-a1**2
dL_dz1=dL_da1*da1_dz1
dL_dw1=dL_dz1*x
dL_db1=dL_dz1
print('dL/dw2 =',dL_dw2)
print('dL/db2 =',dL_db2)
print('dL/dw1 =',dL_dw1)
print('dL/db1 =',dL_db1)

Each early gradient is a product of local derivatives along the path from that parameter to the loss.

## 3. Visualize the computational graph

In [ ]:
nodes={'x':(0,1),'w1':(0,2),'z1':(1.3,1.5),'a1':(2.6,1.5),'w2':(2.6,2.6),'z2':(4,1.5),'ŷ':(5.3,1.5),'L':(6.6,1.5)}
edges=[('x','z1'),('w1','z1'),('z1','a1'),('a1','z2'),('w2','z2'),('z2','ŷ'),('ŷ','L')]
plt.figure(figsize=(13,4))
for name,(px,py) in nodes.items(): plt.scatter(px,py,s=1300); plt.text(px,py,name,ha='center',va='center',fontsize=12)
for a,b in edges:
    x1,y1=nodes[a]; x2,y2=nodes[b]; plt.annotate('',xy=(x2-.12,y2),xytext=(x1+.12,y1),arrowprops=dict(arrowstyle='->'))
plt.xlim(-.6,7.2); plt.ylim(.2,3.2); plt.axis('off'); plt.title('Forward values → ; backward gradients ← through the same graph'); plt.show()

## 4. Numerical gradient check

In [ ]:
def forward_loss(w1_,w2_):
    z1_=w1_*x+b1; a1_=np.tanh(z1_); z2_=w2_*a1_+b2; p=sigmoid(z2_); return -(y*np.log(p)+(1-y)*np.log(1-p))
eps=1e-5
num_dw1=(forward_loss(w1+eps,w2)-forward_loss(w1-eps,w2))/(2*eps)
num_dw2=(forward_loss(w1,w2+eps)-forward_loss(w1,w2-eps))/(2*eps)
print('analytical dw1:',dL_dw1,' numerical:',num_dw1)
print('analytical dw2:',dL_dw2,' numerical:',num_dw2)
print('dw1 check:',np.allclose(dL_dw1,num_dw1,rtol=1e-4,atol=1e-6))
print('dw2 check:',np.allclose(dL_dw2,num_dw2,rtol=1e-4,atol=1e-6))

Gradient checking compares analytical backpropagation with finite differences. Agreement is a strong debugging signal that the derivative implementation is correct.